# Next Best Action

Turn the omnichannel channel-plan state into one governed recommendation per HCP-account row. Generate a candidate menu, gate it, resolve by precedence, rank inside the gates with response and uplift, attach a recommendation contract, then add a bandit, an off-policy check, and the experiment that would settle the policy question. The carried case is HCP0280 at account ACC089.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from ch09_nba.scripts.next_best_action import run_analysis  # noqa: E402

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
results = run_analysis(ROOT)
print(f"Recommendations: {len(results['recommendations'])}")
print(f"Candidates: {len(results['action_candidates'])}")


Recommendations: 158
Candidates: 1106


## Candidate set


In [2]:
candidates = results["action_candidates"]
trace = candidates.loc[candidates.npi.eq("9000000280")]
print(trace[[
    "candidate_action", "eligible", "policy_precedence", "reason_code"
]].to_string(index=False))


           candidate_action  eligible  policy_precedence                                                  reason_code
                  No action      True                 90                  No higher-precedence eligible action passed
           Access follow-up     False                 10                   Account evidence points to access friction
         Field conversation     False                 20       Priority HCP-account row with permitted field capacity
         Program invitation     False                 25   Prior live-program attendance supports a repeat invitation
             Approved email      True                 30  Available email frequency with a priority or digital signal
Continue responsive content      True                 40 Meaningful digital response without a higher-priority action
                    Monitor      True                 80    Eligible HCP-account row without a stronger action signal


![Figure 9.1. HCP0280 starts with 7 candidates; the gate removes access, field, and program actions, then precedence selects approved email and records the rejected alternatives. Synthetic data.](assets/figures/figure_9_1_decision_engine.svg)

*Figure 9.1. HCP0280 starts with 7 candidates; the gate removes access, field, and program actions, then precedence selects approved email and records the rejected alternatives. Synthetic data.*


## Eligibility gates


In [3]:
reasons = [
    "Suppressed", "Access route first", "Not priority",
    "No live-program signal", "Passed",
]
gate_summary = results["gate_summary"].set_index("ineligibility_reason")
print(gate_summary.loc[reasons].reset_index().to_string(index=False))


  ineligibility_reason  blocked_candidates
            Suppressed                 276
    Access route first                 175
          Not priority                  69
No live-program signal                  36
                Passed                 397


## Policy precedence


In [4]:
summary = results["recommendation_summary"].copy()
summary["mean_predicted_response"] = summary.mean_predicted_response.round(3)
print(summary.to_string(index=False))


         recommended_action  recommendations  review_required  mean_predicted_response
                  No action               46                0                    0.510
           Access follow-up               35               35                    0.665
         Program invitation               35                0                    0.670
                    Monitor               20                0                    0.506
             Approved email               13                0                    0.634
Continue responsive content                6                0                    0.695
         Field conversation                3                3                    0.615


![Figure 9.2. The engine reduces 1,106 candidates to 397 eligible candidates and 158 selected actions, with most released rows going to no action, access follow-up, and program invitation. Synthetic data.](assets/figures/figure_9_2_recommendation_funnel.svg)

*Figure 9.2. The engine reduces 1,106 candidates to 397 eligible candidates and 158 selected actions, with most released rows going to no action, access follow-up, and program invitation. Synthetic data.*


## Reward design: response and uplift

The response and uplift models are fixed inputs from the channel analysis. The NBA engine uses them for a different decision: which reward should rank scarce eligible actions after gates and precedence. Response ranks likely engagement. Uplift ranks expected incremental change.


In [5]:
print(results["reward_overlap"].to_string(index=False))


                               metric  value
Promotional-eligible HCP-account rows  51.00
          Spearman response vs uplift  -0.78
       Top-20 shared by both rankings   1.00
      Top-20 only in response ranking  19.00


In [6]:
reward = results["reward_candidates"].copy()
print(reward[[
    "npi", "candidate_action", "predicted_response",
    "estimated_uplift", "rank_by_response", "rank_by_uplift"
]].head(6).round(3).to_string(index=False))


       npi   candidate_action  predicted_response  estimated_uplift  rank_by_response  rank_by_uplift
9000000128 Program invitation               0.844             0.039                 1              45
9000000239 Program invitation               0.839             0.041                 2              43
9000000204 Program invitation               0.831             0.024                 3              50
9000000232 Program invitation               0.828             0.036                 4              48
9000000650     Approved email               0.803             0.052                 5              37
9000000406 Program invitation               0.802             0.056                 6              36


![Figure 9.3. Promotional-eligible rows split into high-response sure things and higher-uplift persuadable rows; HCP0204 shows why response alone can waste a scarce program slot. Synthetic data.](assets/figures/figure_9_3_reward_design.svg)

*Figure 9.3. Promotional-eligible rows split into high-response sure things and higher-uplift persuadable rows; HCP0204 shows why response alone can waste a scarce program slot. Synthetic data.*


## The recommendation contract


In [7]:
recommendations = results["recommendations"]
row = recommendations.loc[recommendations.npi.eq("9000000280")].iloc[0]
for field in [
    "recommendation_id", "account_id", "recommended_action",
    "recommended_channel", "reason_code", "expected_result",
    "measurement_hook", "recommendation_date", "expires_on",
    "review_required",
]:
    print(f"{field}: {row[field]}")


recommendation_id: NBA00131
account_id: ACC089
recommended_action: Approved email
recommended_channel: Email
reason_code: Available email frequency with a priority or digital signal
expected_result: Deliver approved content and earn a click
measurement_hook: Delivery and click
recommendation_date: 2025-02-28 00:00:00
expires_on: 2025-03-14 00:00:00
review_required: False


## Rejected-alternative audit


In [8]:
print(results["audit_summary"].to_string(index=False))


             candidate_status  candidates
                   Ineligible         709
Eligible but lower precedence         239
                     Selected         158


In [9]:
audit = results["candidate_audit"]
trace = audit.loc[audit.npi.eq("9000000280")]
print(trace[[
    "candidate_action", "candidate_status", "policy_precedence"
]].to_string(index=False))


           candidate_action              candidate_status  policy_precedence
                  No action Eligible but lower precedence                 90
           Access follow-up                    Ineligible                 10
         Field conversation                    Ineligible                 20
         Program invitation                    Ineligible                 25
             Approved email                      Selected                 30
Continue responsive content Eligible but lower precedence                 40
                    Monitor Eligible but lower precedence                 80


![Figure 9.4. Each candidate action is split into selected, eligible but lower precedence, or ineligible status, making rejected alternatives visible by action type. Synthetic data.](assets/figures/figure_9_4_candidate_audit.svg)

*Figure 9.4. Each candidate action is split into selected, eligible but lower precedence, or ineligible status, making rejected alternatives visible by action type. Synthetic data.*


## Lifecycle and expiration


In [10]:
print(results["expiration_analysis"].to_string(index=False))


                      metric  value
  Median days between events 12.000
    Mean days between events 17.300
Share of gaps within 14 days  0.573
Share of gaps within 30 days  0.828


![Figure 9.5. The cumulative refresh curve shows that 57% of HCP-account event gaps close within 14 days and 83% close within 30 days. Synthetic data.](assets/figures/figure_9_5_expiration.svg)

*Figure 9.5. The cumulative refresh curve shows that 57% of HCP-account event gaps close within 14 days and 83% close within 30 days. Synthetic data.*


## Exploration with a contextual bandit


In [11]:
exploration = results["thompson_exploration"].copy()
print(exploration[[
    "context_bucket", "logged_action", "snapshots", "posterior_mean",
    "posterior_sd", "explore_share"
]].to_string(index=False))


    context_bucket      logged_action  snapshots  posterior_mean  posterior_sd  explore_share
Digital-responsive Field conversation         96           0.633         0.048          0.736
Digital-responsive     Approved email        141           0.594         0.041          0.264
Digital-responsive          No action         79           0.383         0.054          0.000


In [12]:
cold = results["thompson_cold_start"].copy()
print(cold[[
    "context_bucket", "logged_action", "snapshots", "posterior_mean",
    "posterior_sd", "explore_share"
]].to_string(index=False))


    context_bucket      logged_action  snapshots  posterior_mean  posterior_sd  explore_share
Digital-responsive     Approved email         35           0.568         0.080          0.372
Digital-responsive Field conversation         20           0.545         0.104          0.307
Digital-responsive          No action         20           0.545         0.104          0.320


![Figure 9.6. For digital-responsive rows, cold-start action posteriors overlap and Thompson sampling explores; with full history, field conversation separates from no action while email remains plausible. Synthetic data.](assets/figures/figure_9_6_thompson.svg)

*Figure 9.6. For digital-responsive rows, cold-start action posteriors overlap and Thompson sampling explores; with full history, field conversation separates from no action while email remains plausible. Synthetic data.*


## Off-policy evaluation of an alternative policy


The current NBA policy uses suppression, access, promotion, monitoring, and no-action precedence to release one action per HCP-account row. The digital-first variant keeps the same gates but puts approved email ahead of field conversation for priority rows with a high response score.

Each historical decision row has context at the decision date, the action the old policy released, and the meaningful response observed in the following recommendation window. Off-policy evaluation replays the candidate policy on that same context. Matched rows carry observed reward for the candidate action; disagreements show where the log has no observed reward for what the candidate would have done.


![Figure 9.7. Off-policy evaluation replays a candidate policy on historical NBA decisions and estimates the expected response for the full strategy before deployment. Synthetic data.](assets/figures/figure_9_7_ope_policy_replay.svg)

*Figure 9.7. Off-policy evaluation replays a candidate policy on historical NBA decisions and estimates the expected response for the full strategy before deployment. Synthetic data.*


IPS and SNIPS use matched historical rewards with propensity weights. The direct method scores candidate actions with a reward model. Doubly robust estimation starts with the reward model and adds a weighted correction from matched logged rewards.


In [13]:
policy = results["off_policy_evaluation"].copy()
policy["estimated_response_rate"] = policy.estimated_response_rate.map(
    lambda x: f"{x:.1%}"
)
policy["effective_sample_size"] = policy.effective_sample_size.round(1)
print(policy.to_string(index=False))


       policy      estimator estimated_response_rate  matched_snapshots  effective_sample_size
logged_policy on_policy_mean                   57.3%               1422                 1422.0
digital_first            ips                   54.7%               1263                 1262.1
digital_first          snips                   56.7%               1263                 1262.1
digital_first  doubly_robust                   57.4%               1263                 1262.1


## The experiment that would settle it


In [14]:
print(results["experiment_design"].to_string(index=False))


                           parameter    value
              Baseline response rate    0.598
           Minimum detectable effect    0.050
                               Power    0.800
                     Two-sided alpha    0.050
   Required HCP-account rows per arm 1474.000
Eligible HCP-account rows this cycle  112.000
           Cycles to reach both arms   27.000


## Conclusion

The engine turns a dated state into one auditable action per HCP-account row: a candidate menu, hard eligibility gates, precedence, response and uplift ranking inside the gates, and a recommendation contract with every rejected alternative recorded. The bandit explores only where the posteriors overlap, off-policy evaluation screens a new policy before a live test, and the experiment design shows the eligible population must be pooled across cycles to reach power. For HCP0280 the governed recommendation is approved email.
